In [ ]:

import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
import numpy as np

# Load the training and test data
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/DSEval/datasets/02_cardiovascular_diseases/train.csv'
test_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/DSEval/datasets/02_cardiovascular_diseases/test.csv'

train_df = pd.read_csv(train_data_path)
test_df = pd.read_csv(test_data_path)

# Display the first few rows of each dataset
train_df.head(), test_df.head()


(  General_Health  ... FriedPotato_Consumption
 0           Good  ...                    24.0
 1           Good  ...                     8.0
 2      Very Good  ...                    16.0
 3      Excellent  ...                     8.0
 4      Very Good  ...                     2.0
 
 [5 rows x 19 columns],
   General_Health  ... FriedPotato_Consumption
 0      Very Good  ...                     4.0
 1           Good  ...                     2.0
 2      Very Good  ...                     4.0
 3      Very Good  ...                     8.0
 4           Fair  ...                    30.0
 
 [5 rows x 19 columns])

In [ ]:


# Step 1: Drop rows with missing values and duplicates
train_df = train_df.dropna().drop_duplicates()
test_df = test_df.dropna().drop_duplicates()

# Verify the number of rows left in the datasets
train_df.shape, test_df.shape


((49415, 19), (12355, 19))

In [ ]:


# Step 2: Feature construction - Create BMI_Category feature
def categorize_bmi(bmi):
    if bmi < 18.5:
        return 'Underweight'
    elif bmi < 25:
        return 'Normal weight'
    elif bmi < 30:
        return 'Overweight'
    else:
        return 'Obesity'

# Assuming there is a 'BMI' column in the datasets
train_df['BMI_Category'] = train_df['BMI'].apply(categorize_bmi).astype('category')
test_df['BMI_Category'] = test_df['BMI'].apply(categorize_bmi).astype('category')

# Verify the new feature
train_df[['BMI', 'BMI_Category']].head(), test_df[['BMI', 'BMI_Category']].head()



##active_line5##
(     BMI   BMI_Category
 0  43.40        Obesity
 1  19.53  Normal weight
 2  23.30  Normal weight
 3  27.12     Overweight
 4  20.66  Normal weight,
      BMI   BMI_Category
 0  25.75     Overweight
 1  36.05        Obesity
 2  29.18     Overweight
 3  24.41  Normal weight
 4  37.76        Obesity)

In [ ]:



# Step 3: Model Training
# Assuming 'Heart_Disease' is the target variable
target = 'Heart_Disease'

# Separate features and target
X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]

# Convert categorical variables to numerical using one-hot encoding
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Ensure the same columns in both datasets
common_columns = X_train.columns.intersection(X_test.columns)
X_train = X_train[common_columns]
X_test = X_test[common_columns]

# Train a RandomForestClassifier
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train)

# Step 4: Model Evaluation
# Make predictions on the test set
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Compute the area under the ROC curve
auc_roc = roc_auc_score(y_test, y_pred_proba)

auc_roc


np.float64(0.8059259466557597)